[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/01_instrumentation_trace_storage.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/01_instrumentation_trace_storage.ipynb)

# Experiment 01: Instrumentation & Trace Storage
**Phase 1 Exploration**: Validating asynchronous span emission, immutable trace creation, schema constraints, and storage repository interfaces under simulated application loads.

## 1. Environment Setup
Install `nirizan` directly from the main branch or setup local dev dependencies.

In [1]:
import sys

# Install NiriZan and dependencies silently when running in Colab/Kaggle
!pip install -q pydantic>=2.7 "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan"

from datetime import datetime, timezone
import time
from uuid import uuid4
import asyncio

from nirizan.instrumentation.spans import Span, SpanKind, Trace
from pydantic import ValidationError

print("✅ NiriZan successfully loaded!")

✅ NiriZan successfully loaded!


## 2. Validating `Span` Model Contracts
Spans are the atomic unit of instrumentation[cite: 6]. We test:
1. Span creation across all `SpanKind` enums (`PLANNING`, `RETRIEVAL`, `TOOL_USE`, `GENERATION`)[cite: 6].
2. Strict frozen immutability (`frozen=True`)[cite: 6].
3. Attribute primitive typing constraints[cite: 6].

In [2]:
trace_id = uuid4()
now = datetime.now(timezone.utc)

# 1. Create a retrieval span
retrieval_span = Span(
    span_id=uuid4(),
    trace_id=trace_id,
    kind=SpanKind.RETRIEVAL,
    name="qdrant_vector_search",
    started_at=now,
    ended_at=now,
    attributes={"top_k": 5, "vector_dim": 1536, "hybrid_search": True},
    input_payload="What is continuous evaluation?",
    output_payload="Doc 1: Continuous evaluation infrastructure...",
)

print(f"Created Span ID: {retrieval_span.span_id}")
print(f"Attributes: {retrieval_span.attributes}")

# 2. Test Immutability
try:
    retrieval_span.name = "modified_name"  # type: ignore
except ValidationError as e:
    print("\n✅ Immutability verified: Cannot modify frozen Span instance!")

Created Span ID: 2493873d-cd03-4ff6-a8e1-f31d067140b8
Attributes: {'top_k': 5, 'vector_dim': 1536, 'hybrid_search': True}

✅ Immutability verified: Cannot modify frozen Span instance!


## 3. Assembling a `Trace`
A `Trace` is an ordered collection of spans belonging to a single application invocation[cite: 6].
We test:
1. `trace_id` validation across child spans[cite: 6].
2. Filtering spans by `SpanKind` via `spans_of_kind()`[cite: 6].

In [3]:
generation_span = Span(
    span_id=uuid4(),
    trace_id=trace_id,
    kind=SpanKind.GENERATION,
    name="llm_generate_answer",
    started_at=now,
    ended_at=now,
    attributes={"model": "gpt-4o", "temperature": 0.2},
    input_payload="Context: ... Prompt: What is continuous evaluation?",
    output_payload="Continuous evaluation is an engineering layer...",
)

# Create Trace
trace = Trace(
    trace_id=trace_id,
    application_name="production_rag_service",
    spans=[retrieval_span, generation_span],
    created_at=now,
)

print(f"Trace ID: {trace.trace_id}")
print(f"Total Spans: {len(trace.spans)}")
print(f"Retrieval Spans: {len(trace.spans_of_kind(SpanKind.RETRIEVAL))}")
print(f"Generation Spans: {len(trace.spans_of_kind(SpanKind.GENERATION))}")

# Test trace_id mismatch assertion
mismatched_span = Span(
    span_id=uuid4(),
    trace_id=uuid4(),  # Different trace_id!
    kind=SpanKind.PLANNING,
    name="query_planner",
    started_at=now,
    ended_at=now,
)

try:
    Trace(
        trace_id=trace_id,
        application_name="invalid_trace_app",
        spans=[mismatched_span],
        created_at=now,
    )
except ValidationError:
    print("\n✅ Trace ID validation verified: Rejected mismatched span!")

Trace ID: 9b209e61-8f23-41c7-b968-9a45c80f79c9
Total Spans: 2
Retrieval Spans: 1
Generation Spans: 1

✅ Trace ID validation verified: Rejected mismatched span!


## 4. Measuring Tracing Overhead (Latency Benchmark)
Instrumentation must **never** block the application's request/response path[cite: 6].
We simulate async background trace exportation to verify minimal latency impact.

In [4]:
class MockAsyncExporter:
    """Simulates an asynchronous background collector export."""

    async def export(self, trace: Trace) -> None:
        # Simulate background network/storage latency without blocking caller
        await asyncio.sleep(0.05)


async def simulate_application_request(exporter: MockAsyncExporter):
    start_time = time.perf_counter()

    # Application execution simulation
    t_id = uuid4()
    n = datetime.now(timezone.utc)
    s = Span(
        span_id=uuid4(),
        trace_id=t_id,
        kind=SpanKind.GENERATION,
        name="rag_response",
        started_at=n,
        ended_at=n,
    )
    tr = Trace(trace_id=t_id, application_name="benchmark_app", spans=[s], created_at=n)

    # Fire-and-forget background task for trace emission
    asyncio.create_task(exporter.export(tr))

    elapsed_ms = (time.perf_counter() - start_time) * 1000
    return elapsed_ms


async def run_benchmark():
    exporter = MockAsyncExporter()
    latencies = []

    for _ in range(1000):
        latency = await simulate_application_request(exporter)
        latencies.append(latency)

    avg_latency = sum(latencies) / len(latencies)
    print(f"⚡ Tracing Latency Overhead across 1,000 runs:")
    print(f"   Average Overhead: {avg_latency:.4f} ms per request")
    print(f"   Max Single-Request Overhead: {max(latencies):.4f} ms (Non-blocking verified)")


await run_benchmark()

⚡ Tracing Latency Overhead across 1,000 runs:
   Average Overhead: 0.0218 ms per request
   Max Single-Request Overhead: 1.1679 ms (Non-blocking verified)


Emitting traces via fire-and-forget background tasks imposes practically zero latency penalty on the caller application path (averaging $\approx 0.015\text{ ms}$ per request). The non-blocking constraint is verified.



---



## 5. Prototyping Context-Aware `Tracer` (`contextvars`)
Manual instantiation of UUIDs and timestamps is error-prone. A `Tracer` uses Python's built-in `contextvars` module to track the active `trace_id` and `current_span_id` across nested function calls automatically without requiring explicit argument passing.

We test:
1. Implicit context propagation across nested execution contexts.
2. Automatic `parent_span_id` linking between parent and child spans.

In [5]:
import contextvars
from contextlib import asynccontextmanager
from datetime import datetime, timezone
import functools
from typing import Any, AsyncGenerator, Callable, Optional
from uuid import UUID, uuid4

# Context variables for thread/async-safe execution tracing
_current_trace_id: contextvars.ContextVar[Optional[UUID]] = contextvars.ContextVar(
    "current_trace_id", default=None
)
_current_span_id: contextvars.ContextVar[Optional[UUID]] = contextvars.ContextVar(
    "current_span_id", default=None
)


class PrototypeTracer:
    """Manages span lifecycle and context propagation."""

    def __init__(self, application_name: str = "demo_app"):
        self.application_name = application_name
        self.active_spans: list[Span] = []

    @asynccontextmanager
    async def start_span(
        self,
        name: str,
        kind: SpanKind,
        attributes: dict[str, Any] | None = None,
        input_payload: str | None = None,
    ) -> AsyncGenerator[UUID, None]:
        # 1. Resolve or initialize Trace ID
        trace_id = _current_trace_id.get()
        if trace_id is None:
            trace_id = uuid4()
            _current_trace_id.set(trace_id)

        # 2. Automatically capture parent span ID from current context
        parent_span_id = _current_span_id.get()

        # 3. Create new span & set start timestamp
        span_id = uuid4()
        started_at = datetime.now(timezone.utc)

        # Set this span as the current active span context
        token_span = _current_span_id.set(span_id)

        try:
            yield span_id
        finally:
            ended_at = datetime.now(timezone.utc)

            # Create immutable span record on exit
            completed_span = Span(
                span_id=span_id,
                trace_id=trace_id,
                parent_span_id=parent_span_id,
                kind=kind,
                name=name,
                started_at=started_at,
                ended_at=ended_at,
                attributes=attributes or {},
                input_payload=input_payload,
            )
            self.active_spans.append(completed_span)

            # Restore previous span context
            _current_span_id.reset(token_span)

    def get_assembled_trace(self) -> Trace:  # Added `self` parameter
        trace_id = _current_trace_id.get()
        return Trace(
            trace_id=trace_id or uuid4(),
            application_name=self.application_name,
            spans=list(self.active_spans),
            created_at=datetime.now(timezone.utc),
        )


print("✅ PrototypeTracer with contextvars initialized successfully!")

✅ PrototypeTracer with contextvars initialized successfully!


## 6. Prototyping Developer SDK (`@trace_span` Decorator)
To keep application code clean, we prototype a decorator SDK wrapper around `Tracer`.

We test:
1. End-to-end tracing of a nested RAG pipeline execution tree.
2. Verification of implicit parent-child relationship (`root` -> `retrieval` + `generation`).

In [6]:
tracer = PrototypeTracer(application_name="rag_sdk_experiment")


def trace_span(kind: SpanKind, name: str | None = None):
    """Decorator to automatically instrument async functions."""

    def decorator(func: Callable):
        span_name = name or func.__name__

        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            input_str = str(args[0]) if args else str(kwargs)
            async with tracer.start_span(name=span_name, kind=kind, input_payload=input_str):
                return await func(*args, **kwargs)

        return wrapper

    return decorator


# --- Instrumented Application Services ---


@trace_span(kind=SpanKind.RETRIEVAL, name="qdrant_vector_fetch")
async def fetch_documents(query: str) -> list[str]:
    await asyncio.sleep(0.01)  # Simulate DB latency
    return ["Doc 1: Continuous evaluation...", "Doc 2: Telemetry instrumentation..."]


@trace_span(kind=SpanKind.GENERATION, name="openai_llm_call")
async def generate_response(query: str, docs: list[str]) -> str:
    await asyncio.sleep(0.02)  # Simulate LLM inference latency
    return "Continuous evaluation infrastructure tracks real-time performance."


@trace_span(kind=SpanKind.PLANNING, name="rag_orchestrator")
async def run_rag_pipeline(user_query: str) -> str:
    docs = await fetch_documents(user_query)
    answer = await generate_response(user_query, docs)
    return answer


# --- Run Experiment ---


async def run_sdk_verification():
    answer = await run_rag_pipeline("What is continuous evaluation?")
    assembled_trace = tracer.get_assembled_trace()

    print(f"Pipeline Result: '{answer}'\n")
    print(f"📊 Trace Verification (Trace ID: {assembled_trace.trace_id}):")
    print(f"   Total Spans Captured: {len(assembled_trace.spans)}")

    # Identify Root Span vs Child Spans
    root_spans = [s for s in assembled_trace.spans if s.parent_span_id is None]
    child_spans = [s for s in assembled_trace.spans if s.parent_span_id is not None]

    print(f"   Root Spans: {len(root_spans)} ({root_spans[0].name})")
    print(f"   Child Spans: {len(child_spans)}")

    # Assertions
    root_span = root_spans[0]
    for child in child_spans:
        assert child.parent_span_id == root_span.span_id, (
            f"Child span {child.name} parent_span_id mismatch!"
        )
        assert child.trace_id == root_span.trace_id, f"Child span {child.name} trace_id mismatch!"
        print(
            f"   └─ Span '{child.name}' ({child.kind.value}) correctly linked to Parent '{root_span.name}'"
        )

    print("\n✅ Automatic context propagation & SDK parent-child linking verified!")


await run_sdk_verification()

Pipeline Result: 'Continuous evaluation infrastructure tracks real-time performance.'

📊 Trace Verification (Trace ID: cd6c0e0f-7d84-45f8-8aa9-a65e6adf6c27):
   Total Spans Captured: 3
   Root Spans: 1 (rag_orchestrator)
   Child Spans: 2
   └─ Span 'qdrant_vector_fetch' (retrieval) correctly linked to Parent 'rag_orchestrator'
   └─ Span 'openai_llm_call' (generation) correctly linked to Parent 'rag_orchestrator'

✅ Automatic context propagation & SDK parent-child linking verified!


## 7. Prototyping Storage Models (`nirizan.storage.models`)
Runtime `Trace` and `Span` models are optimized for in-memory context tracking. For persistence, we need storage-oriented models that handle serialization to database records (e.g., ISO timestamps, stringified JSON payloads, indexed UUIDs).

We test:
1. Serialization of runtime `Trace` objects into storage records.
2. Lossless deserialization back into runtime domain models.

In [7]:
import json
from typing import Any, Optional
from uuid import UUID
from datetime import datetime
from pydantic import BaseModel, Field


class SpanRecord(BaseModel):
    """Database record representation of a Span."""

    span_id: str
    trace_id: str
    parent_span_id: Optional[str] = None
    kind: str
    name: str
    started_at: str
    ended_at: str
    attributes_json: str = "{}"
    input_payload: Optional[str] = None
    output_payload: Optional[str] = None

    @classmethod
    def from_span(cls, span: Span) -> "SpanRecord":
        return cls(
            span_id=str(span.span_id),
            trace_id=str(span.trace_id),
            parent_span_id=str(span.parent_span_id) if span.parent_span_id else None,
            kind=span.kind.value,
            name=span.name,
            started_at=span.started_at.isoformat(),
            ended_at=span.ended_at.isoformat(),
            attributes_json=json.dumps(span.attributes),
            input_payload=span.input_payload,
            output_payload=span.output_payload,
        )

    def to_span(self) -> Span:
        """Inverse of from_span. Needed so the repository can hand back a
        real Span/Trace at its public boundary (see docs/contracts.md),
        keeping this record shape an internal storage detail only."""
        return Span(
            span_id=UUID(self.span_id),
            trace_id=UUID(self.trace_id),
            parent_span_id=UUID(self.parent_span_id) if self.parent_span_id else None,
            kind=SpanKind(self.kind),
            name=self.name,
            started_at=datetime.fromisoformat(self.started_at),
            ended_at=datetime.fromisoformat(self.ended_at),
            attributes=json.loads(self.attributes_json),
            input_payload=self.input_payload,
            output_payload=self.output_payload,
        )


class TraceRecord(BaseModel):
    """Database record representation of a complete Trace."""

    trace_id: str
    application_name: str
    created_at: str
    spans: list[SpanRecord] = Field(default_factory=list)

    @classmethod
    def from_trace(cls, trace: Trace) -> "TraceRecord":
        return cls(
            trace_id=str(trace.trace_id),
            application_name=trace.application_name,
            created_at=trace.created_at.isoformat(),
            spans=[SpanRecord.from_span(s) for s in trace.spans],
        )

    def to_trace(self) -> Trace:
        """Inverse of from_trace. See SpanRecord.to_span for why this exists."""
        return Trace(
            trace_id=UUID(self.trace_id),
            application_name=self.application_name,
            created_at=datetime.fromisoformat(self.created_at),
            spans=[s.to_span() for s in self.spans],
        )


# --- Quick Verification ---
sample_record = TraceRecord.from_trace(trace)
print(f"\u2705 Serialized Trace ID: {sample_record.trace_id}")
print(f"   Spans in Record: {len(sample_record.spans)}")
print(f"   First Span Kind: {sample_record.spans[0].kind}")

# Round-trip check
restored = sample_record.to_trace()
assert restored.trace_id == trace.trace_id
assert len(restored.spans) == len(trace.spans)
print(f"\u2705 Round-trip TraceRecord -> Trace verified!")

✅ Serialized Trace ID: 9b209e61-8f23-41c7-b968-9a45c80f79c9
   Spans in Record: 2
   First Span Kind: retrieval
✅ Round-trip TraceRecord -> Trace verified!


## 8. Prototyping Trace Repository (`nirizan.storage.trace_repository`)
The `TraceRepository` interface handles durable persistence. We implement an async SQLite repository using `sqlite3` and Python's `asyncio.to_thread` executor to verify query contracts without adding external database dependencies to this experiment.

We test:
1. Creating trace and span relational tables.
2. Persisting `TraceRecord` hierarchies.
3. Retrieving traces by `trace_id` with all spans restored.

In [8]:
from abc import ABC, abstractmethod
import sqlite3
import asyncio
from typing import Optional
from uuid import UUID
from datetime import datetime


class BaseTraceRepository(ABC):
    """Abstract interface for trace storage engines.

    Matches docs/contracts.md's TraceRepository interface: operates on
    Trace directly at the public boundary. TraceRecord/SpanRecord stay an
    internal storage detail, never crossing this interface.
    """

    @abstractmethod
    async def save(self, trace: Trace) -> None:
        pass

    @abstractmethod
    async def get(self, trace_id: UUID) -> Optional[Trace]:
        pass


class SQLiteTraceRepository(BaseTraceRepository):
    """In-memory or file-backed SQLite persistence engine for testing storage contracts."""

    def __init__(self, db_path: str = ":memory:"):
        self.db_path = db_path
        # Keep a single persistent connection open for in-memory databases
        self._conn = sqlite3.connect(self.db_path, check_same_thread=False)
        self._conn.row_factory = sqlite3.Row
        self._init_db()

    def _init_db(self) -> None:
        with self._conn:
            self._conn.executescript(
                """
                CREATE TABLE IF NOT EXISTS traces (
                    trace_id TEXT PRIMARY KEY,
                    application_name TEXT NOT NULL,
                    created_at TEXT NOT NULL
                );

                CREATE TABLE IF NOT EXISTS spans (
                    span_id TEXT PRIMARY KEY,
                    trace_id TEXT NOT NULL,
                    parent_span_id TEXT,
                    kind TEXT NOT NULL,
                    name TEXT NOT NULL,
                    started_at TEXT NOT NULL,
                    ended_at TEXT NOT NULL,
                    attributes_json TEXT NOT NULL,
                    input_payload TEXT,
                    output_payload TEXT,
                    FOREIGN KEY(trace_id) REFERENCES traces(trace_id)
                );
                """
            )

    async def save(self, trace: Trace) -> None:
        # Convert to the internal record shape right here, at the storage
        # boundary, matching src/nirizan/storage/trace_repository.py.
        record = TraceRecord.from_trace(trace)

        def _insert():
            with self._conn:
                self._conn.execute(
                    "INSERT OR REPLACE INTO traces (trace_id, application_name, created_at) VALUES (?, ?, ?)",
                    (record.trace_id, record.application_name, record.created_at),
                )
                for span in record.spans:
                    self._conn.execute(
                        """
                        INSERT OR REPLACE INTO spans (
                            span_id, trace_id, parent_span_id, kind, name,
                            started_at, ended_at, attributes_json, input_payload, output_payload
                        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """,
                        (
                            span.span_id,
                            span.trace_id,
                            span.parent_span_id,
                            span.kind,
                            span.name,
                            span.started_at,
                            span.ended_at,
                            span.attributes_json,
                            span.input_payload,
                            span.output_payload,
                        ),
                    )

        await asyncio.to_thread(_insert)

    async def get(self, trace_id: UUID) -> Optional[Trace]:
        trace_id_str = str(trace_id)

        def _query():
            trace_row = self._conn.execute(
                "SELECT * FROM traces WHERE trace_id = ?", (trace_id_str,)
            ).fetchone()

            if not trace_row:
                return None

            span_rows = self._conn.execute(
                "SELECT * FROM spans WHERE trace_id = ?", (trace_id_str,)
            ).fetchall()

            spans = [
                SpanRecord(
                    span_id=row["span_id"],
                    trace_id=row["trace_id"],
                    parent_span_id=row["parent_span_id"],
                    kind=row["kind"],
                    name=row["name"],
                    started_at=row["started_at"],
                    ended_at=row["ended_at"],
                    attributes_json=row["attributes_json"],
                    input_payload=row["input_payload"],
                    output_payload=row["output_payload"],
                )
                for row in span_rows
            ]

            return TraceRecord(
                trace_id=trace_row["trace_id"],
                application_name=trace_row["application_name"],
                created_at=trace_row["created_at"],
                spans=spans,
            )

        record = await asyncio.to_thread(_query)
        # Convert back to Trace right at the boundary, mirroring
        # TraceRecord.to_trace() in src/nirizan/storage/models.py.
        return record.to_trace() if record is not None else None


# --- Test Repository Persistence ---
async def verify_repository():
    repo = SQLiteTraceRepository()

    await repo.save(trace)
    fetched = await repo.get(trace.trace_id)

    assert fetched is not None, "Failed to retrieve saved trace!"
    assert fetched.trace_id == trace.trace_id, "Trace ID mismatch!"
    assert len(fetched.spans) == len(trace.spans), "Span count mismatch!"

    print(f"\u2705 SQLite Trace Repository verified!")
    print(f"   Successfully persisted and retrieved Trace ID: {fetched.trace_id}")
    print(f"   Restored {len(fetched.spans)} spans from relational database tables.")


await verify_repository()

✅ SQLite Trace Repository verified!
   Successfully persisted and retrieved Trace ID: 9b209e61-8f23-41c7-b968-9a45c80f79c9
   Restored 2 spans from relational database tables.


## 9. Prototyping Trace Collector Orchestrator (`nirizan.orchestrator.collector`)
The `TraceCollector` acts as an async ingestion buffer. It receives traces emitted by application exporters, pushes them to an internal `asyncio.Queue`, and persists them to the storage repository via a background worker task without blocking caller requests.

We test:
1. Non-blocking async queue ingestion.
2. Background worker processing and storage persistence.
3. Full end-to-end flow: `Tracer` $\rightarrow$ `Exporter` $\rightarrow$ `Collector` $\rightarrow$ `Repository`.

In [9]:
from typing import Protocol


class TraceSink(Protocol):
    """The shape TraceCollector needs from a repository, and nothing more.

    Matches src/nirizan/orchestrator/collector.py: orchestrator/ never
    imports storage/ directly (see docs/import-boundaries.md). Any object
    with an async save(trace) method satisfies this structurally, no
    inheritance or import required. SQLiteTraceRepository above satisfies
    this without knowing this Protocol exists.
    """

    async def save(self, trace: Trace) -> None: ...


class TraceCollector:
    """Ingestion orchestrator that buffers incoming traces for persistence."""

    def __init__(self, repository: TraceSink):
        self.repository = repository
        self.queue: asyncio.Queue[Trace] = asyncio.Queue()
        self._worker_task: Optional[asyncio.Task] = None
        self._running = False

    async def start(self) -> None:
        """Start the background worker processor."""
        self._running = True
        self._worker_task = asyncio.create_task(self._process_queue())

    async def stop(self) -> None:
        """Flush remaining queue items and stop worker."""
        self._running = False
        await self.queue.join()
        if self._worker_task:
            self._worker_task.cancel()

    async def enqueue_trace(self, trace: Trace) -> None:
        """Non-blocking trace ingestion."""
        await self.queue.put(trace)

    async def _process_queue(self) -> None:
        while self._running or not self.queue.empty():
            try:
                trace = await asyncio.wait_for(self.queue.get(), timeout=0.1)
                # No TraceRecord conversion here: the collector hands the
                # repository a Trace directly and knows nothing about how
                # storage represents it internally.
                await self.repository.save(trace)
                self.queue.task_done()
            except asyncio.TimeoutError:
                continue


class CollectorExporter(MockAsyncExporter):
    """Bridge exporter that pushes completed traces to a TraceCollector."""

    def __init__(self, collector: TraceCollector):
        self.collector = collector

    async def export(self, trace: Trace) -> None:
        await self.collector.enqueue_trace(trace)


# --- Run End-to-End Phase 1 Integration Test ---
async def run_full_phase1_pipeline_test():
    # 1. Setup storage & orchestrator collector
    repo = SQLiteTraceRepository()
    collector = TraceCollector(repository=repo)
    await collector.start()

    # 2. Wire Tracer SDK to Collector Exporter
    collector_exporter = CollectorExporter(collector)
    e2e_tracer = PrototypeTracer(application_name="e2e_production_app")

    # 3. Simulate application workload with tracing
    async with e2e_tracer.start_span("orchestrator_pipeline", SpanKind.PLANNING):
        async with e2e_tracer.start_span("qdrant_vector_lookup", SpanKind.RETRIEVAL):
            await asyncio.sleep(0.01)

        async with e2e_tracer.start_span("llm_generation_step", SpanKind.GENERATION):
            await asyncio.sleep(0.01)

    # 4. Export completed trace to collector
    generated_trace = e2e_tracer.get_assembled_trace()
    await collector_exporter.export(generated_trace)

    # 5. Stop collector (flushes queue)
    await collector.stop()

    # 6. Verify trace in database
    retrieved_trace = await repo.get(generated_trace.trace_id)

    assert retrieved_trace is not None, "Trace missing from storage database!"
    assert retrieved_trace.application_name == "e2e_production_app"
    assert len(retrieved_trace.spans) == 3

    print("\U0001f680 End-to-End Phase 1 Integration Test Successful!")
    print(f"   Trace ID: {retrieved_trace.trace_id}")
    print(f"   Application: {retrieved_trace.application_name}")
    print(f"   Persisted Spans: {len(retrieved_trace.spans)}")
    for span in retrieved_trace.spans:
        print(
            f"     \u2514\u2500 [{span.kind.value.upper()}] {span.name} (ID: {str(span.span_id)[:8]}...)"
        )


await run_full_phase1_pipeline_test()

🚀 End-to-End Phase 1 Integration Test Successful!
   Trace ID: c86748ee-2c20-4f3e-ad96-fb880b7d04c4
   Application: e2e_production_app
   Persisted Spans: 3
     └─ [RETRIEVAL] qdrant_vector_lookup (ID: 1f0f41e4...)
     └─ [GENERATION] llm_generation_step (ID: e0c2f69c...)
     └─ [PLANNING] orchestrator_pipeline (ID: cd667964...)
